Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_movielens
from src.content_based import ContentBasedRecommender
from src.popularity import PopularityRecommender
from src.hybrid import HybridRecommender
from src.random_baseline import RandomRecommender
from src.evaluation import (
    compare_models,
    compare_user_recommendations,
    evaluate_by_user_activity,
    train_test_split_by_time,
    user_genre_profile,
)

Load Dataset

In [ ]:
ratings, movies, users = load_movielens()

Initialize the Model

In [ ]:
model = ContentBasedRecommender(movies)

Generate Recommendations

In [ ]:
recommendations = model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(recommendations)

## Popularity-Based Recommendations

The baseline recommends movies with at least `min_ratings` ratings and the highest share of positive (>= threshold) ratings. Every user gets the same list, minus movies they've already rated.

In [ ]:
popularity_model = PopularityRecommender(movies, min_ratings=50)

popular_recommendations = popularity_model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(popular_recommendations)

## Hybrid Recommendations

Combines both scores: `score = alpha * content_score + (1 - alpha) * popularity_score`. Both scores are rescaled to 0-1 first. `alpha=1` is pure content-based and `alpha=0` is pure popularity.

In [ ]:
hybrid_model = HybridRecommender(movies, alpha=0.5, min_ratings=50)

hybrid_recommendations = hybrid_model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(hybrid_recommendations)

## Evaluation

Each user's most recent 20% of ratings are held out as a test set. Each model only sees the training ratings, and we check how many of its top-10 recommendations are movies the user went on to rate 4 or higher.

- **precision@10**: share of the 10 recommendations the user liked
- **recall@10**: share of the user's liked held-out movies that were recommended
- **hit_rate@10**: share of users with at least one liked movie in their top 10
- **ndcg@10**: rank-weighted gain rewarding liked movies placed near the top
- **mrr@10**: mean reciprocal rank of the first liked movie in the top 10
- **coverage**: share of the whole catalog recommended to at least one user
- **personalization**: average `1 - cosine_similarity` across users' top-10 lists (0 means identical lists, 1 means disjoint)

The random recommender is a floor: any useful model should clearly beat it on precision, recall, hit rate, ndcg, and mrr (random will always win on coverage).

In [ ]:
train, test = train_test_split_by_time(ratings, test_fraction=0.2)

print(f"Train ratings: {len(train):,}")
print(f"Test ratings:  {len(test):,}")

In [ ]:
models = {
    "Random (floor)": RandomRecommender(movies, seed=0),
    "Content-based": ContentBasedRecommender(movies),
    "Popularity": PopularityRecommender(movies, min_ratings=50),
    "Hybrid (alpha=0.5)": HybridRecommender(movies, alpha=0.5, min_ratings=50),
}

comparison = compare_models(models, train, test, k=10, threshold=4)
display(comparison.round(4))

In [ ]:
plot_metrics = [
    "precision@10", "recall@10", "hit_rate@10",
    "ndcg@10", "coverage", "personalization",
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, metric in zip(axes.flat, plot_metrics):
    comparison[metric].plot.bar(ax=ax)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Recommender comparison")
plt.tight_layout()
plt.show()

### Choosing alpha

Run the hybrid with different weights from 0.0 (popularity) to 1.0 (content-based) to see how accuracy, coverage, and personalization change.

In [ ]:
alphas = [round(i * 0.1, 1) for i in range(11)]

sweep = compare_models(
    {a: HybridRecommender(movies, alpha=a, min_ratings=50) for a in alphas},
    train, test, k=10, threshold=4
)
sweep.index.name = "alpha"
display(sweep.round(4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, metric in zip(axes.flat, plot_metrics):
    ax.plot(sweep.index, sweep[metric], marker="o")
    ax.set_title(metric)
    ax.set_xlabel("alpha")

plt.suptitle("Hybrid performance by alpha (0 = popularity, 1 = content-based)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(sweep["coverage"], sweep["precision@10"], marker="o")
for a_val, row in sweep.iterrows():
    plt.annotate(f"{a_val:.1f}", (row["coverage"], row["precision@10"]), textcoords="offset points", xytext=(5, 4), fontsize=8)
plt.title("Precision@10 vs. Coverage across alpha")
plt.xlabel("coverage")
plt.ylabel("precision@10")
plt.tight_layout()
plt.show()

In [ ]:
best_alpha = float(sweep["precision@10"].idxmax())
print(f"Best alpha by precision@10: {best_alpha}")
print(f"Hybrid at best alpha:  precision@10={sweep.loc[best_alpha, 'precision@10']:.4f}, ndcg@10={sweep.loc[best_alpha, 'ndcg@10']:.4f}")
print(f"Content-based alone:   precision@10={sweep.loc[1.0, 'precision@10']:.4f}, ndcg@10={sweep.loc[1.0, 'ndcg@10']:.4f}")
print(f"Popularity alone:      precision@10={sweep.loc[0.0, 'precision@10']:.4f}, ndcg@10={sweep.loc[0.0, 'ndcg@10']:.4f}")

### Robustness check

Re-run the comparison with different hold-out sizes to check that the ranking of the models does not depend on one split.

In [ ]:
robustness_prec = {}
robustness_ndcg = {}
for fraction in [0.1, 0.2, 0.3]:
    tr, te = train_test_split_by_time(ratings, test_fraction=fraction)
    res = compare_models(
        {
            "Random (floor)": RandomRecommender(movies, seed=0),
            "Content-based": ContentBasedRecommender(movies),
            "Popularity": PopularityRecommender(movies, min_ratings=50),
            f"Hybrid (alpha={best_alpha})": HybridRecommender(movies, alpha=best_alpha, min_ratings=50),
        },
        tr, te, k=10, threshold=4
    )
    robustness_prec[fraction] = res["precision@10"]
    robustness_ndcg[fraction] = res["ndcg@10"]

robustness_prec = pd.DataFrame(robustness_prec)
robustness_prec.columns.name = "test_fraction"
print("precision@10:")
display(robustness_prec.round(4))

robustness_ndcg = pd.DataFrame(robustness_ndcg)
robustness_ndcg.columns.name = "test_fraction"
print("ndcg@10:")
display(robustness_ndcg.round(4))

### Sensitivity to min_ratings

Check how changing `min_ratings` affects accuracy and catalog coverage for Popularity and Hybrid.

In [ ]:
min_ratings_values = [10, 25, 50, 100, 200]
sens_models = {}
for mr in min_ratings_values:
    sens_models[f"Popularity (min={mr})"] = PopularityRecommender(movies, min_ratings=mr)
    sens_models[f"Hybrid (alpha={best_alpha}, min={mr})"] = HybridRecommender(movies, alpha=best_alpha, min_ratings=mr)

sens_df = compare_models(sens_models, train, test, k=10, threshold=4)
display(sens_df[["precision@10", "ndcg@10", "hit_rate@10", "coverage", "personalization"]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for prefix in ["Popularity", "Hybrid"]:
    sub = sens_df.loc[[i for i in sens_df.index if i.startswith(prefix)]]
    axes[0].plot(min_ratings_values, sub["precision@10"], marker="o", label=prefix)
    axes[1].plot(min_ratings_values, sub["coverage"], marker="o", label=prefix)

axes[0].set_title("precision@10 vs. min_ratings")
axes[0].set_xlabel("min_ratings")
axes[0].legend()
axes[1].set_title("coverage vs. min_ratings")
axes[1].set_xlabel("min_ratings")
axes[1].legend()
plt.tight_layout()
plt.show()

### Performance by user activity

Compare the models across light, medium, and heavy users based on how many ratings each user had in the training set.

In [ ]:
seg_models = {
    "Random (floor)": RandomRecommender(movies, seed=0),
    "Content-based": ContentBasedRecommender(movies),
    "Popularity": PopularityRecommender(movies, min_ratings=50),
    f"Hybrid (alpha={best_alpha})": HybridRecommender(movies, alpha=best_alpha, min_ratings=50),
}
activity_df = evaluate_by_user_activity(seg_models, train, test, k=10, threshold=4, quantiles=3)
display(activity_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["precision@10", "ndcg@10"]):
    piv = activity_df.pivot(index="segment", columns="model", values=metric).loc[["Light", "Medium", "Heavy"]]
    piv[list(seg_models.keys())].plot.bar(ax=ax)
    ax.set_title(f"{metric} by user activity")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

### Sample user comparisons

Look at top-10 lists for three users with different rating patterns: one focused mainly on a single genre, one heavy user who rates many genres, and one light user.

In [ ]:
profiles = [user_genre_profile(ratings, movies, int(u), threshold=4) for u in sorted(ratings["user_id"].unique())]
prof_df = pd.DataFrame([
    {
        "user_id": p["user_id"],
        "total_ratings": p["total_ratings"],
        "positive_ratings": p["positive_ratings"],
        "n_genres_liked": len(p["genre_counts"]),
        "top_genre_share": (p["genre_counts"].iloc[0] / p["positive_ratings"]) if p["positive_ratings"] > 0 else 0.0,
    }
    for p in profiles
])

concentrated_uid = int(prof_df[prof_df["positive_ratings"] >= 20].sort_values(["top_genre_share", "user_id"], ascending=[False, True]).iloc[0]["user_id"])
diverse_uid = int(prof_df[prof_df["total_ratings"] >= 200].sort_values(["n_genres_liked", "total_ratings", "user_id"], ascending=[False, False, True]).iloc[0]["user_id"])
light_uid = int(prof_df[(prof_df["total_ratings"] == prof_df["total_ratings"].min()) & (prof_df["positive_ratings"] >= 8)].sort_values("user_id").iloc[0]["user_id"])

sample_models = {
    "Popularity": PopularityRecommender(movies, min_ratings=50),
    "Content-based": ContentBasedRecommender(movies),
    f"Hybrid (alpha={best_alpha})": HybridRecommender(movies, alpha=best_alpha, min_ratings=50),
}

for label, uid in [("Concentrated user", concentrated_uid), ("Diverse heavy user", diverse_uid), ("Light user", light_uid)]:
    p = user_genre_profile(ratings, movies, uid, threshold=4)
    top_genres = ", ".join(f"{g} ({c})" for g, c in p["genre_counts"].head(5).items())
    print(f"{label} (user_id={uid}): {p['total_ratings']} ratings, {p['positive_ratings']} positive, mean={p['mean_rating']:.2f}")
    print(f"Top genres: {top_genres}")
    table = compare_user_recommendations(sample_models, ratings, movies, user_id=uid, n=10, threshold=4)
    display(table)
    pop_set = set(table["Popularity"])
    cb_set = set(table["Content-based"])
    hy_set = set(table[f"Hybrid (alpha={best_alpha})"])
    print(f"Overlap: Content/Popularity={len(cb_set & pop_set)}/10, Hybrid/Popularity={len(hy_set & pop_set)}/10, Hybrid/Content={len(hy_set & cb_set)}/10\n")

### Results

**Model comparison.** [Which model had the highest precision@10, recall@10 and hit_rate@10? Give the numbers. How far above the random floor was each model?]

**Coverage.** [Which model recommended the widest share of the catalog, and which the narrowest? What does that trade-off mean for users?]

**Choosing alpha.** [Which alpha gave the best precision@10? Describe how the metrics changed as alpha went from 0 (popularity) to 1 (content-based).]

**Robustness.** [Did the ranking of the models stay the same for test_fraction 0.1, 0.2 and 0.3?]

**Sensitivity and user groups.** [Summarize how min_ratings, user activity levels, and the sample user comparisons behaved.]

**Hypothesis.** [We hypothesized personalized recommendations would beat popularity, and the hybrid would balance the two. Was that supported?]